# dfs-three-set-toposort — worked example 1: Topological sort of a four-node diamond DAG

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dfs-three-set-toposort`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The three-set DFS topological sort uses a `perm` set (fully processed nodes), a `temp` set (nodes currently on the recursion stack), and a `result` list that receives each node in post-order — after all its descendants. Because a node is appended only after all its children finish, the list comes out in deps-first order. Reversing it gives the order suitable for a backward pass.

## Worked solution

We have four nodes: A → B, A → C, B → D, C → D. We call `visit(A)`.

**Step 1.** `A` is not in `perm` or `temp`, so add `A` to `temp`. Children of `A` are `[B, C]`; call `visit(B)` first.

**Step 2.** `B` is fresh; add to `temp`. Children of `B`: `[D]`; call `visit(D)`.

**Step 3.** `D` is fresh; add to `temp`. `D` has no children. Remove `D` from `temp`, add to `perm`, append `D` to result. Result: `[D]`.

**Step 4.** Back in `visit(B)`: all children done. Remove `B` from `temp`, add to `perm`, append `B`. Result: `[D, B]`.

**Step 5.** Back in `visit(A)`, next child is `C`. `D` is already in `perm` — when `visit(D)` is called from `visit(C)`, the first check returns immediately. Remove `C` from temp, append. Result: `[D, B, C]`.

**Step 6.** Back in `visit(A)`: append `A`. Result: `[D, B, C, A]`. Root `A` is last, all deps come before their dependents.

In [ ]:
def topological_sort_diamond():
    # Build a simple diamond: A -> B, A -> C, B -> D, C -> D
    children = {'A': ['B', 'C'], 'B': ['D'], 'C': ['D'], 'D': []}

    result = []
    perm = set()
    temp = set()

    def visit(node):
        nid = id(node)
        if nid in perm:
            return
        if nid in temp:
            raise ValueError(f'Cycle at {node!r}')
        temp.add(nid)
        for child in children[node]:
            visit(child)
        temp.remove(nid)
        perm.add(nid)
        result.append(node)

    visit('A')
    return result

ordering = topological_sort_diamond()
print('Topological order (deps first):', ordering)
print('Root A is last:', ordering[-1] == 'A')
print('D appears before B and C:', ordering.index('D') < ordering.index('B') and ordering.index('D') < ordering.index('C'))